In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

#functions
def load_patent_distributions(parquet_file, start_year=2000, end_year=2020):
    con = duckdb.connect()
    df = con.execute(f"""
        SELECT
            vc_backed,
            npl_ratio,
            LN(total_claims + 1) AS log_claims
        FROM '{parquet_file}'
        WHERE grant_year BETWEEN {start_year} AND {end_year};
    """).df()
    df['vc_status'] = df['vc_backed'].map({1: 'vc-backed', 0: 'non-vc'})
    return df

def configure_plot_style():
    plt.rcParams.update({
        'font.size': 10,
        'font.family': 'sans-serif',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'grid.alpha': 0.3,
        'grid.color': '#000000',
        'axes.edgecolor': '#000000'
    })

def plot_science_intensity_density(df, output_path, palette):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)

    sns.kdeplot(
        data=df,
        x='npl_ratio',
        hue='vc_status',
        fill=True,
        common_norm=False,
        palette=palette,
        ax=ax,
        alpha=0.3,
        linewidth=1.1
    )

    ax.set_title('Empirical Distribution of Science Intensity', fontsize=11, fontweight='bold', pad=12, color='#000000')
    ax.set_xlabel('NPL Ratio [npl / (backward citations + npl)]', fontsize=10, color='#000000')
    ax.set_ylabel('Density', fontsize=10, color='#000000')
    ax.set_xlim(0, 1.0)
    ax.grid(True, linestyle=':')

    caption = (
        "figure 2a: Empirical distribution of science intensity comparing vc-backed and non-vc patents (2000–2020).\n"
        "science intensity is defined as the ratio of non-patent literature citations to total backward citations."
    )
    plt.figtext(0.5, -0.12, caption, wrap=True, horizontalalignment='center', fontsize=8.5, color='#000000')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"saved figure 2a: {output_path}")

def plot_patent_scope_density(df, output_path, palette):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)

    sns.kdeplot(
        data=df,
        x='log_claims',
        hue='vc_status',
        fill=True,
        common_norm=False,
        palette=palette,
        ax=ax,
        alpha=0.3,
        linewidth=1.1
    )

    ax.set_title('Empirical Distribution of Patent Scope', fontsize=11, fontweight='bold', pad=12, color='#000000')
    ax.set_xlabel('Log(total claims + 1)', fontsize=10, color='#000000')
    ax.set_ylabel('Density', fontsize=10, color='#000000')
    ax.grid(True, linestyle=':')

    caption = (
        "Figure 2B: Empirical distribution of patent scope comparing VC-backed and non-VC backed patents (2000-2020).\n"
        "Patent scope is proxied by the natural logarithm of total claim count plus one to adjust for skewness."
    )
    plt.figtext(0.5, -0.12, caption, wrap=True, horizontalalignment='center', fontsize=8.5, color='#000000')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"saved figure 2b: {output_path}")

#figure 2

#setup paths and load data
drive_mount_point = os.path.join(os.sep, "content", "drive")
drive.mount(drive_mount_point, force_remount=False)

drive_data_dir = os.path.join(drive_mount_point, "MyDrive", "data")
parquet_path = os.path.join(drive_data_dir, "cleaned_patent_panel.parquet")

df_plot = load_patent_distributions(parquet_path, start_year=2000, end_year=2020)


configure_plot_style()
palette = {'vc-backed': '#4da6ff', 'non-vc': '#005a32'}

#figure 2a: science intensity density
fig_2a_path = os.path.join(drive_data_dir, "figure2a_science_intensity.png")
plot_science_intensity_density(df_plot, fig_2a_path, palette)

#figure 2b: patent scope density
fig_2b_path = os.path.join(drive_data_dir, "figure2b_patent_scope.png")
plot_patent_scope_density(df_plot, fig_2b_path, palette)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved figure 2a: /content/drive/MyDrive/data/figure2a_science_intensity.png
saved figure 2b: /content/drive/MyDrive/data/figure2b_patent_scope.png
